In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statistics as stats
from pathlib import Path
import os

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muqaddasejaz/european-flights-dataset")

print("Path to dataset files:", path)

In [ ]:
import os

os.listdir(path)

In [ ]:
df = pd.read_csv(os.path.join(path, "European Flights Dataset.csv"))
df.head()

## 1. Check data

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df["APT_NAME"].nunique()

In [ ]:
df["STATE_NAME"].nunique()

## 2. Cleaning data

In [ ]:
ifr_cols = [
    "FLT_DEP_IFR_2",
    "FLT_ARR_IFR_2",
    "FLT_TOT_IFR_2"
]

df[ifr_cols] = df[ifr_cols].fillna(0)

In [ ]:
df["FLT_DATE"] = pd.to_datetime(df["FLT_DATE"], errors="coerce")

In [ ]:
df = df.rename(columns={
    "FLT_DEP_1": "departure_flights",
    "FLT_ARR_1": "arrival_flights",
    "FLT_TOT_1": "total_flights",

    "FLT_DEP_IFR_2": "departure_ifr",
    "FLT_ARR_IFR_2": "arrival_ifr",
    "FLT_TOT_IFR_2": "total_ifr",

    "APT_NAME": "airport_name",
    "APT_ICAO": "airport_icao",
    "STATE_NAME": "country"
})

df["airport_country"] = (
    df["airport_name"] +
    " (" +
    df["country"] +
    ")"
)
df.head()

In [ ]:
df = df.drop(columns=["Pivot Label"])

df = df.drop(columns=["MONTH_MON"])

In [ ]:
df["month_name"] = df["FLT_DATE"].dt.month_name()
df.head()

## 3. EDA

### 3.1 Which airport is the busiest?

In [ ]:
total_fligths_by_airport = df.groupby("airport_country")["total_flights"].sum()
# print("Total flights:", total_fligths)
total_fligths_by_airport.sort_values(ascending=False).head(20)

In [ ]:
colors = plt.cm.cividis(
    np.linspace(0, 1, 20)
)

plt.figure(figsize=(12, 9))
plt.bar(total_fligths_by_airport.sort_values(ascending=False).head(20).index,
        total_fligths_by_airport.sort_values(ascending=False).head(20).values,
        color=colors)
plt.xticks(rotation=75, ha='right')
plt.title("Top 20 Airports by Total Flights")
plt.xlabel("Airport Name")
plt.ylabel("Total Flights")
plt.tight_layout()

os.makedirs("plots", exist_ok=True)
filepath = os.path.join("plots", "total_fligths_by_airport.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight", facecolor="white")

### 3.2 Which country has the busiest flight activity?

In [ ]:
country_flights = (
    df.groupby("country")["total_flights"]
    .sum()
    .sort_values(ascending=False)
)
country_flights.head(20)

In [ ]:
colors = plt.cm.plasma(
    np.linspace(0, 1, 20)
)

plt.figure(figsize=(12, 9))
plt.bar(country_flights.sort_values(ascending=False).head(20).index,
        country_flights.sort_values(ascending=False).head(20).values,
        color=colors)
plt.xticks(rotation=90)
plt.title("Top 20 Countries by Total Flights")
plt.xlabel("Country")
plt.ylabel("Total Flights")
plt.tight_layout()

os.makedirs("plots", exist_ok=True)
filepath = os.path.join("plots", "total_country_flights.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight", facecolor="white")

In [ ]:
target_countries = ["United Kingdom", "Germany", "France", "Spain"]

# Total flights per negara
country_flights = (
    df[df["country"].isin(target_countries)]
    .groupby("country")["total_flights"]
    .sum()
    .reset_index()
    .sort_values(by="total_flights", ascending=False)
)

# Total seluruh penerbangan
total_all_flights = df["total_flights"].sum()

# Hitung persentase kontribusi
country_flights["percentage"] = (
    country_flights["total_flights"] / total_all_flights * 100
)

# Rapihin persen
country_flights["percentage"] = (
    country_flights["percentage"].round(2)
)

country_flights

Insight :
- Most Dutch flights are concentrated at just one airport, namely Amsterdam-Schipol Airport.
- Flight activity in Europe is mostly in the western European region, such as France, Germany, UK and Spain. These countries account for almost half of all flights in Europe (48.37%)

### 3.3 Which month is the busiest?

In [ ]:
month_order = [
    "January", "February", "March",
    "April", "May", "June",
    "July", "August", "September",
    "October", "November", "December"
]

monthly_flights = (
    df.groupby("month_name")["total_flights"]
    .sum()
    .reindex(month_order)
)

monthly_flights

In [ ]:
colors = plt.cm.plasma(
    np.linspace(0, 1, 20)
)

plt.figure(figsize=(12, 9))
plt.bar(monthly_flights.index,
        monthly_flights.values,
        color=colors)
# plt.xticks(rotation=45)
plt.title("Monthly Flights")
plt.xlabel("Month")
plt.ylabel("Total Flights")
plt.tight_layout()

os.makedirs("plots", exist_ok=True)
filepath = os.path.join("plots", "monthly_flights.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight", facecolor="white")
plt.ylabel("Total Flights")
plt.tight_layout()

### 3.4 Top 15 Countries Fligth Traffic by Month

In [ ]:
top_15 = (
    df.groupby('country')['total_flights']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .index
)

filtered_df = df[df['country'].isin(top_15)]

pivot = filtered_df.pivot_table(
    values='total_flights',
    index='country',
    columns='month_name',
    aggfunc='sum'
)

month_order = [
    'January', 'February', 'March',
    'April', 'May', 'June',
    'July', 'August', 'September',
    'October', 'November', 'December'
]

pivot = pivot[month_order]

plt.figure(figsize=(14,8))

sns.heatmap(
    pivot,
    annot=True,
    fmt='.0f'
)

plt.title('Top 15 Countries Flight Traffic by Month')
plt.xlabel('Month')
plt.ylabel('Country')

filepath = os.path.join("plots", "heatmap_country_vs_monthly_flights.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight", facecolor="white")
plt.tight_layout()

### 3.5 Box plot total flights per country

In [ ]:
from matplotlib.lines import Line2D

top_10 = (
    df.groupby('country')['total_flights']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

filtered_df = df[df['country'].isin(top_10)]

plt.figure(figsize=(14,8))

box = plt.boxplot(
    [filtered_df[filtered_df['country'] == country]['total_flights'] for country in top_10],
    labels=top_10, patch_artist=True, showmeans=True, medianprops={'color': 'red', 'linewidth': 2, 'linestyle': '--', 'label': 'Median'},
    whiskerprops={'color': 'green', 'linewidth': 2, 'linestyle': '--', 'label': 'Whiskers'}, 
    meanprops={
        "marker" : "D",
        "markerfacecolor" : "red",
        "markeredgecolor" : "black",
        "markersize" : 10,
        "label" : "Mean"
    }
)

legend_elements = [

    Line2D(
        [0],
        [0],
        color='red',
        linestyle='--',
        linewidth=2,
        label='Median'
    ),

    Line2D(
        [0],
        [0],

        marker='D',
        color='white',
        markerfacecolor='red',
        markeredgecolor='black',
        markersize=10,
        label='Mean'
    ),

    Line2D(
        [0],
        [0],
        color='green',
        linestyle='--',
        linewidth=2,
        label='Whiskers'
    )
]

for b in box['boxes']:
    b.set(color='blue', linewidth=2)
    b.set(facecolor='lightblue')

plt.title('Flight Distribution by Country (top 10)')
plt.xlabel('Country')
plt.ylabel('Total Flights')
plt.legend(handles=legend_elements, loc='upper right')
plt.xticks(rotation=45)
plt.yscale('log')
plt.tight_layout()

filepath = os.path.join("plots", "boxplot_country_flight_distribution.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

### 3.6 Trend per year

In [ ]:
flight_per_year = (
    df.groupby("YEAR")["total_flights"]
    .sum()
    .reset_index()
)

# Pisah data
before_pandemic = flight_per_year[flight_per_year["YEAR"] <= 2019]
pandemic = flight_per_year[flight_per_year["YEAR"] >= 2020]

plt.figure(figsize=(12, 6))

# Sebelum pandemi
plt.plot(
    before_pandemic["YEAR"],
    before_pandemic["total_flights"],
    marker="o",
    linewidth=3,
    label="Before Pandemic"
)

# Saat pandemi
plt.plot(
    pandemic["YEAR"],
    pandemic["total_flights"],
    marker="o",
    linewidth=3,
    linestyle="--",
    label="Pandemic Era"
)

# Highlight area pandemi
plt.axvspan(2020, 2022, alpha=0.2)

# Annotation
plt.annotate(
    "COVID-19 Impact",
    xy=(2020, 7500000),
    xytext=(2018.5, 12000000),
    arrowprops=dict(arrowstyle="->", lw=3),
    fontsize=11
)

# Styling
plt.title("Total Flights per Year", fontsize=16, weight="bold")
plt.xlabel("Year", fontsize=12)
plt.ylabel("Total Flights", fontsize=12)

plt.xticks(flight_per_year["YEAR"])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

filepath = os.path.join("plots", "total_flights_per_year.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight", facecolor="white")

plt.show()

### 3.7 Monthly Total Flights Over Time

In [ ]:
monthly_flights = (
    df.groupby(
        pd.Grouper(
            key="FLT_DATE",
            freq="ME"
        )
    )["total_flights"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(12, 6))
plt.plot(
    monthly_flights["FLT_DATE"],
    monthly_flights["total_flights"],
    linewidth=2.5,
    color="brown",
    label="Monthly Total Flights"
)
plt.title("Monthly Total Flights Over Time", fontsize=16, weight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Total Flights", fontsize=12)
plt.grid(alpha=0.3)

plt.axvspan(
    pd.Timestamp("2020-03-01"),
    pd.Timestamp("2021-12-31"),
    color="red",
    alpha=0.15,
    label="COVID-19 Period"
)

monthly_flights["rolling_mean"] = (
    monthly_flights["total_flights"]
    .rolling(3)
    .mean()
)

plt.plot(
    monthly_flights["FLT_DATE"],
    monthly_flights["rolling_mean"],
    linewidth=3,
    linestyle="--",
    label="3-Month Rolling Mean"
)

min_idx = monthly_flights["total_flights"].idxmin()

min_x = monthly_flights.loc[min_idx, "FLT_DATE"]
min_y = monthly_flights.loc[min_idx, "total_flights"]

plt.annotate(
    "COVID Collapse",
    xy=(min_x, min_y),
    xytext=(pd.Timestamp("2019-01-01"), 1000000),
    arrowprops=dict(arrowstyle="->", lw=2)
)

import matplotlib.dates as mdates

plt.gca().xaxis.set_major_locator(
    mdates.YearLocator()
)

plt.gca().xaxis.set_major_formatter(
    mdates.DateFormatter('%Y')
)

plt.legend()

plt.tight_layout()

filepath = os.path.join("plots", "monthly_flights_over_time.png")
plt.savefig(filepath, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

### 3.8 Which countries' flights are most affected by Covid-19?

In [ ]:
import plotly.express as px

top_8_countries = (
    df.groupby("country")["total_flights"]
    .sum()
    .nlargest(8)
    .index
)

filtered_df = df[
    df["country"].isin(top_8_countries)
]

monthly_country = (
    filtered_df.groupby(
        ["country",
         pd.Grouper(
             key="FLT_DATE",
             freq="ME"
         )]
    )["total_flights"]
    .sum()
    .reset_index()
)

fig = px.line(
    monthly_country,
    x="FLT_DATE",
    y="total_flights",
    color="country",
    title="Top 8 Countries Flight Trends Over Time",
    
    hover_data={
        "country": True,
        "FLT_DATE": False,
        "total_flights": ":,.0f"
    }
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Total Flights",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

filepath = os.path.join(
    "plots",
    "top8_countries_flights.html"
)

fig.write_html(filepath)

### 3.9 Peak Month Per Year

In [ ]:
monthly_total = (
    df.groupby(
        pd.Grouper(
            key="FLT_DATE",
            freq="ME"
        )
    )["total_flights"]
    .sum()
    .reset_index()
)

monthly_total["YEAR"] = (
    monthly_total["FLT_DATE"].dt.year
)

monthly_total["MONTH"] = (
    monthly_total["FLT_DATE"].dt.month_name()
)

peak_months = (
    monthly_total.loc[
        monthly_total.groupby("YEAR")[
            "total_flights"
        ].idxmax()
    ]
)

print(
    peak_months[
        ["YEAR", "MONTH", "total_flights"]
    ]
)

| Year | Peak Month | Total Flights |
|------|------|------|
| 2016 | July | 1,479,337 |
| 2017 | July | 1,546,209 |
| 2018 | July | 1,655,518 |
| 2019 | July | 1,675,015 |
| 2020 | January | 1,214,726 |
| 2021 | August | 1,150,868 |
| 2022 | May | 1,338,338 |

### 3.10 Which airports are most affected by Covid-19 flights?

In [ ]:
top_8_airports = (
    df.groupby("airport_country")["total_flights"]
    .sum()
    .nlargest(8)
    .index
)

filtered_df = df[
    df["airport_country"].isin(top_8_airports)
]

monthly_airpot_country = (
    filtered_df.groupby(
        ["airport_country",
         pd.Grouper(
             key="FLT_DATE",
             freq="ME"
         )]
    )["total_flights"]
    .sum()
    .reset_index()
)

fig = px.line(
    monthly_airpot_country,
    x="FLT_DATE",
    y="total_flights",
    color="airport_country",
    title="Top 8 airports Flight Trends Over Time",
    
    hover_data={
        "airport_country": True,
        "FLT_DATE": False,
        "total_flights": ":,.0f"
    }
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Total Flights",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

filepath = os.path.join(
    "plots",
    "top8_airports_flights.html"
)

fig.write_html(filepath)